# Muon variants: loss vs wall-clock (depth 8)

Parses the nanochat training logs from the sweep (`logs/slurm/sweep/*.out`).
Wall-clock = nanochat's own `total time` counter: pure training step time, excluding the first 10 steps, evals and compile/startup.
For each variant the LR with the best final val bpb is plotted (change `LR_MODE` to pin one LR instead).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DEPTH = 8
LR_MODE = "best"        # "best" = best final val bpb per variant, or a float / string like 0.02 to pin one LR for all variants
SHOW_ALL_LRS = False    # also draw the non-selected LRs as faint lines

LABELS = {
    "eq":    "MuonEq (nanochat default)",
    "eq_m2": "MuonEq + Muon$^2$",
    "m2":    "Muon$^2$",
    "base":  "Muon (no EQ, no Muon$^2$)",
}
COLORS = {"eq": "tab:blue", "eq_m2": "tab:red", "m2": "tab:green", "base": "tab:gray"}

In [ ]:
FILE_RE  = re.compile(r"sweep_d(\d+)_(eq_m2|eq|m2|base)_lr([0-9.eE+-]+?)-(\d+)\.out$")
STEP_RE  = re.compile(r"^step (\d+)/(\d+) .*?\| loss: ([0-9.]+) .*?\| dt: ([0-9.]+)ms .*?total time: ([0-9.]+)m", re.M)
VAL_RE   = re.compile(r"^Step (\d+) \| Validation bpb: ([0-9.]+)", re.M)

# Find every logs/slurm/sweep dir at or above the cwd (the scripts may have been launched from a subdir like jobs/)
log_dirs = [p / "logs" / "slurm" / "sweep" for p in [Path.cwd(), *Path.cwd().parents]]
log_dirs = [d for d in log_dirs if d.is_dir()]
print("log dirs:", [str(d) for d in log_dirs])

runs = {}   # (depth, variant, lr) -> dict; if a tag has several logs keep the one with the most steps
for d in log_dirs:
    for path in d.glob("*.out"):
        m = FILE_RE.search(path.name)
        if not m or int(m.group(1)) != DEPTH:
            continue
        text = path.read_text(errors="replace")
        steps = STEP_RE.findall(text)
        if not steps:
            continue
        train = pd.DataFrame(steps, columns=["step", "n_iters", "loss", "dt_ms", "time_min"]).astype(float)
        val = pd.DataFrame(VAL_RE.findall(text), columns=["step", "val_bpb"]).astype(float)
        # wall-clock (training time only) at each eval step; the last eval comes after the final train step
        val["time_min"] = np.interp(val["step"], train["step"], train["time_min"])
        key = (int(m.group(1)), m.group(2), float(m.group(3)))
        if key not in runs or len(train) > len(runs[key]["train"]):
            runs[key] = dict(train=train, val=val, path=path, n_iters=int(train["n_iters"].iloc[0]))

rows = []
for (depth, v, lr), r in sorted(runs.items()):
    finished = len(r["train"]) >= r["n_iters"]
    rows.append(dict(variant=v, lr=lr, finished=finished,
                     final_val_bpb=r["val"]["val_bpb"].iloc[-1] if len(r["val"]) else np.nan,
                     final_train_loss=r["train"]["loss"].iloc[-1],
                     train_min=r["train"]["time_min"].iloc[-1],
                     median_ms_per_step=r["train"]["dt_ms"].iloc[10:].median()))
summary = pd.DataFrame(rows)
summary

In [ ]:
# Pick one LR per variant
chosen = {}
for v in LABELS:
    s = summary[(summary.variant == v) & summary.finished]
    if s.empty:
        print(f"no finished runs for {v}"); continue
    if LR_MODE == "best":
        chosen[v] = s.loc[s.final_val_bpb.idxmin(), "lr"]
    else:
        lr = float(LR_MODE)
        if lr in set(s.lr): chosen[v] = lr
        else: print(f"{v}: no finished run at lr={lr}")

summary[summary.apply(lambda r: chosen.get(r.variant) == r.lr, axis=1)].set_index("variant").loc[[v for v in LABELS if v in chosen]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for v, lr in chosen.items():
    r = runs[(DEPTH, v, lr)]
    label = f"{LABELS[v]} (lr={lr:g})"
    axes[0].plot(r["train"]["time_min"], r["train"]["loss"], color=COLORS[v], lw=1.5, label=label)
    axes[1].plot(r["val"]["time_min"], r["val"]["val_bpb"], color=COLORS[v], lw=1.5, marker="o", ms=3, label=label)
    if SHOW_ALL_LRS:
        for (d2, v2, lr2), r2 in runs.items():
            if v2 == v and lr2 != lr and len(r2["train"]) >= r2["n_iters"]:
                axes[0].plot(r2["train"]["time_min"], r2["train"]["loss"], color=COLORS[v], lw=0.6, alpha=0.25)
                axes[1].plot(r2["val"]["time_min"], r2["val"]["val_bpb"], color=COLORS[v], lw=0.6, alpha=0.25)

# zoom the y-axes onto the interesting part (drop the initial high-loss transient)
def zoom(ax, ycol, key, width):
    finals = [runs[(DEPTH, v, lr)][key][ycol].iloc[-1] for v, lr in chosen.items()]
    ax.set_ylim(min(finals) - 0.05, min(finals) + width)

zoom(axes[0], "loss", "train", 1.0)
zoom(axes[1], "val_bpb", "val", 0.25)

axes[0].set_title(f"Training loss (EMA-smoothed), depth {DEPTH}")
axes[0].set_ylabel("train loss (nats/token)")
axes[1].set_title(f"Validation bpb, depth {DEPTH}")
axes[1].set_ylabel("val bpb")
for ax in axes:
    ax.set_xlabel("training wall-clock (min)")
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()